# RAG Application with LlamaIndex (SQL)

## Introduction

In this notebook, we will demonstrate how to use LlamaIndex to query a **SQL Database** using natural language (Text-to-SQL). 

We will use the **Chinook** database, a sample database that represents a digital media store (artists, albums, tracks, invoices, etc.).

### What We'll Build
A system that takes a question like "How many tracks are there?" and:
1. Converts it into a SQL query (`SELECT count(*) FROM tracks`).
2. Executes the query against the database.
3. Uses an LLM to explain the answer.

## Step 1: Install Dependencies

We need `llama-index` and `llama-index-readers-database`.

In [1]:
# Install dependencies
!uv pip install -q llama-index llama-index-readers-database sqlalchemy

## Step 2: Setup Environment

Ensure your OpenAI API key is set.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️ Warning: OPENAI_API_KEY not found. Please set it in your .env file.")
else:
    print("✅ OPENAI_API_KEY loaded")

✅ OPENAI_API_KEY loaded


## Step 3: Connect to Database

We use SQLAlchemy to connect to our SQLite database `Chinook.db`.

In [4]:
from sqlalchemy import create_engine
from llama_index.core import SQLDatabase

# Create engine
engine = create_engine("sqlite:///Chinook.db")

# Wrap engine in LlamaIndex SQLDatabase wrapper
# This allows LlamaIndex to inspect schemas and tables
sql_database = SQLDatabase(engine)

print("✅ Connected to Chinook.db")

✅ Connected to Chinook.db


In [9]:
# run query to show the tables available in the database
print(sql_database.get_usable_table_names())

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


## Step 4: Create Query Engine

We use `NLSQLTableQueryEngine`, which specializes in Natural Language -> SQL -> Result.

> **Note:** We are passing the `sql_database` object we just created.

In [12]:
from llama_index.core.query_engine import NLSQLTableQueryEngine

# Create the query engine
query_engine = NLSQLTableQueryEngine(
    sql_database=sql_database,
    #tables=["albums", "tracks", "artists"] # Optional: limit context to specific tables for better performance
)

print("✅ Query Engine Created")

✅ Query Engine Created


## Step 5: Query the Database

Now we can ask complex questions about our data.

In [13]:
response = query_engine.query("How many Tracks are there?")

print(f"Q: How many Tracks are there?")
print(f"A: {response}")

Q: How many Tracks are there?
A: There are a total of 3503 tracks in the database.


In [14]:
response = query_engine.query("Who are the top 5 artists with the most albums?")

print(f"Q: Who are the top 5 artists with the most albums?")
print(f"A: {response}")

Q: Who are the top 5 artists with the most albums?
A: The top 5 artists with the most albums are Iron Maiden with 21 albums, Led Zeppelin with 14 albums, Deep Purple with 11 albums, U2 with 10 albums, and Metallica with 10 albums.


### Inspect the SQL Query

We can see the actual SQL that LlamaIndex generated.

In [15]:
# View the raw metadata to see the SQL query
print(response.metadata['sql_query'])

SELECT Artist.Name, COUNT(Album.AlbumId) AS AlbumCount
FROM Artist
JOIN Album ON Artist.ArtistId = Album.ArtistId
GROUP BY Artist.Name
ORDER BY AlbumCount DESC
LIMIT 5;


## Conclusion

You've just built a Text-to-SQL system in a few lines of code! LlamaIndex handles the prompt engineering required to translate your question into a valid SQL query based on the database schema.